# Day 9 | Lab 9.2: Langfuse — Open-Source Observability

**Duration:** ~1.5 hours

**Scenario.** A regulated bank's GenAI platform team has standardised on LangSmith for development and staging, but the **production** workloads handle customer PII that cannot leave the bank's own VPC. The team needs a *self-hostable, open-source* observability stack with the same span/trace model. **Langfuse** is the canonical answer. In this lab we wire up Langfuse three different ways, log custom quality scores, sketch a dashboard view, and write down the *production decision rules* for picking LangSmith vs Langfuse.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Explain *when* to reach for Langfuse instead of (or alongside) LangSmith — the decision matrix.
2. Instrument a Python function with the `@observe()` decorator and inspect the resulting trace.
3. Wire the `langfuse.langchain.CallbackHandler` into a LangChain chain so every Runnable is traced.
4. Use the `from langfuse.openai import openai` drop-in wrapper for non-LangChain code paths.
5. Push custom quality scores from an LLM-as-Judge pass back onto the original trace.
6. Reason about Langfuse self-hosting (Docker compose vs Kubernetes), data residency, and HA.

**Tools.** `langfuse>=2.0` · LangChain v1 · OpenAI SDK · Claude Sonnet 4.5 (for LLM-as-Judge).

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


## 1. Install Dependencies

In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langfuse>=2.0' 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-openai>=1.0' 'langchain-anthropic>=0.3'
# !pip install -q openai python-dotenv


## 2. API Key Configuration

Langfuse needs **three** environment variables — public key, secret key, and the host URL. Get them from the **Langfuse Cloud** signup (free tier — see §3) or your self-hosted instance's settings page.

If `LANGFUSE_PUBLIC_KEY` is missing, the rest of the lab still parses but the `@observe`-decorated calls and CallbackHandler will silently no-op. The OpenAI / Anthropic keys drive the actual model calls.

In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely
# on environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


In [ ]:
# Optional: pin the Langfuse host. Defaults to https://cloud.langfuse.com when unset.
import os
os.environ.setdefault('LANGFUSE_HOST', 'https://cloud.langfuse.com')
print('LANGFUSE_HOST →', os.environ['LANGFUSE_HOST'])


## 3. Why Langfuse Alongside LangSmith — Decision Matrix

Both tools solve the same problem (LLM observability) but trade off differently. Most large enterprises end up running **both** — LangSmith in dev/stage, Langfuse self-hosted in prod for the regulated tenants — but the team has to know *why* they picked each.

| Dimension | LangSmith Cloud | Langfuse Cloud | Langfuse Self-Hosted |
|---|---|---|---|
| **Pricing model** | Per-trace seat-based; free dev tier | Free tier, then per-event metered | Free (you pay infra) |
| **License** | Closed-source SaaS | Closed-source for cloud | **MIT** (OSS core) |
| **Self-host** | Enterprise add-on | No | **Yes** — Docker compose, Helm chart |
| **Data residency** | US / EU regions | US / EU regions | **Anywhere you can run Postgres + ClickHouse** |
| **LangChain integration** | Native (auto) | Callback handler | Same callback handler |
| **Non-LangChain SDKs** | Manual `@traceable` | `@observe` + OpenAI wrap | Same |
| **Built-in evals** | Datasets / Evaluators / Experiments | Datasets / Evaluators / Annotations | Same as cloud |
| **Prompt management** | LangSmith Hub | Langfuse Prompt Management | Same as cloud |
| **Multi-tenant isolation** | Workspaces | Projects + RBAC | Projects + RBAC + your own VPC |
| **Best for** | LangChain-first dev teams | Multi-SDK teams who want OSS option | Banks / health / gov / sovereign clouds |

**Bottom line.** Pick Langfuse when (a) you need self-hosting under your own data-residency rules, (b) you have a polyglot stack (Python + JS + native OpenAI + LiteLLM, not just LangChain), or (c) the OSS license is a procurement requirement. Pick LangSmith when you're a LangChain-first shop and the SaaS is approved by your security team.

### 3.1 Trace Structure — the Mental Model

Both LangSmith and Langfuse use a four-noun hierarchy. Memorise it; everything else is plumbing.

```
  Trace            ← one user request / one job / one experiment iteration
  └── Span         ← one work step inside the trace (function call, DB query, retrieval, …)
      └── Span     ← spans nest arbitrarily deep
      └── Generation  ← a *specialised* span representing one LLM call
                       (carries model, prompt, completion, tokens, cost)
  Score            ← a numeric / categorical judgement attached to the trace, span, or generation
                     (e.g. empathy=0.8, faithfulness=PASS, user_thumbs_up=true)
```

Filtering, dashboards, and alerting are all driven off this structure. A user-cost dashboard is *"sum trace.cost where trace.userId = X"*. A regression alert is *"average score.empathy this week vs last week"*. Your data model determines the questions you can answer.

## 4. Get a Langfuse Workspace

Two options — pick one before running the rest of the lab.

### 4.1 Option A — Cloud (5 minutes)

1. Go to **https://cloud.langfuse.com** and sign up with email or GitHub.
2. Create a new **Project** (e.g. `eclerx-day9-lab9.2`).
3. In **Settings → API Keys**, click **Create new API key**.
4. Copy the **Public Key**, **Secret Key**, and confirm the **Host** (US: `https://us.cloud.langfuse.com` · EU: `https://cloud.langfuse.com`).
5. Paste them into your local `.env` so step 2 above shows ✅ Loaded for all three.

### 4.2 Option B — Self-Host on Docker (markdown only — do *not* run on the lab machine)

Production-grade reference (read-only — *do not run during the trainer-led lab*; Docker download is ~3 GB and the lab cohort runs on shared bandwidth):

```bash
# 1. Clone the OSS repo
git clone https://github.com/langfuse/langfuse.git
cd langfuse

# 2. Single-command local stack — Postgres + ClickHouse + Langfuse server
docker compose up -d

# 3. Open http://localhost:3000 → create org → create project → grab API keys
#    Configure your .env with LANGFUSE_HOST=http://localhost:3000
```

For production HA (the bank deployment), the architecture is:

```
 ┌──────────┐    ┌──────────────┐    ┌────────────┐    ┌────────────┐
 │  Apps    │ →  │ Langfuse API │ →  │ Postgres   │    │ ClickHouse │
 │ (LC/JS)  │    │  (Node srvr) │    │ (metadata) │    │ (events)   │
 └──────────┘    └──────────────┘    └────────────┘    └────────────┘
       │              ▲                     ▲                ▲
       │              │                     │                │
       └─→ Worker ────┘     Helm chart provides HA replicas + PVCs
```

Helm chart: https://github.com/langfuse/langfuse-k8s — supports horizontal scaling of the API + worker pools, Postgres via cloud-native operator (CNPG / RDS), and ClickHouse via Altinity operator. Standard production sizing for a mid-size bank: **3 API replicas + 2 workers + Postgres HA + 3-node ClickHouse cluster** handles ~50 M events/day comfortably.

## 5. Three Integration Patterns

Langfuse offers three ways to wire it up. **Use all three** in a real codebase — they are designed to compose, not compete.

| Pattern | Use it when |
|---|---|
| **`@observe()` decorator** | Plain-Python functions / your own glue logic — the most fundamental primitive |
| **LangChain `CallbackHandler`** | Anything wrapped in a LangChain `Runnable` / chain / agent / LangGraph |
| **OpenAI SDK drop-in wrap** | Code paths that call `openai.chat.completions.create` directly (no LangChain) |


### 5.1 The `@observe()` Decorator

Wraps any function — sync or async. Captures inputs, outputs, exceptions, and latency. Nests automatically: a decorated function calling another decorated function produces a parent → child span tree.

In [ ]:
from langfuse import observe, get_client

langfuse = get_client()


@observe()
def lookup_account(account_id: str) -> dict:
    """Pretend we hit a CRM API. Just returns a stub."""
    return {'account_id': account_id, 'tier': 'gold', 'open_tickets': 2}


@observe()
def summarise_account(account_id: str) -> str:
    info = lookup_account(account_id)        # ← becomes a child span
    return f"Account {info['account_id']} ({info['tier']}) — {info['open_tickets']} open tickets."


result = summarise_account('CUST-00042')
print(result)

# Flush so the spans land in Langfuse before the cell returns
langfuse.flush()
print('→ Inspect this trace in your Langfuse project under Traces → recent.')


**What just happened.** The decorator opened a *trace* (the parent), then a *span* per nested call, captured the return value, and flushed it to the Langfuse server. No imports of internal SDK objects, no `try/finally` cleanup — the decorator owns the lifecycle.

### 5.2 The LangChain `CallbackHandler`

For anything wrapped in a LangChain Runnable (chains, agents, LangGraph), pass `CallbackHandler()` in the `config` dict. Every step the runnable executes — LLM call, prompt formatting, retrieval, tool call — appears as a span on the trace.

In [ ]:
from langfuse.langchain import CallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_anthropic import ChatAnthropic

lf_handler = CallbackHandler()

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a concise customer-support copywriter. Reply in <=2 sentences.'),
    ('human',  '{customer_question}'),
])

chain = prompt | ChatAnthropic(model='claude-haiku-4-5-20251001', temperature=0) | StrOutputParser()

answer = chain.invoke(
    {'customer_question': 'How long does international shipping take?'},
    config={'callbacks': [lf_handler], 'metadata': {'lab': '9.2', 'pattern': 'langchain-callback'}},
)
print(answer)
langfuse.flush()


### 5.3 OpenAI SDK Drop-in Wrap

If a code path calls the OpenAI SDK directly (a Bedrock-equivalent wrapper, a non-LangChain microservice, a quick prototype), simply swap the import. Every `openai.chat.completions.create()` call now produces a Langfuse *generation* span automatically.

In [ ]:
# Replace `import openai` with the Langfuse drop-in. Same call signatures, automatic instrumentation.
from langfuse.openai import openai

completion = openai.chat.completions.create(
    model='gpt-5-mini',
    messages=[
        {'role': 'system', 'content': 'You are a triage agent. Reply with just the urgency: low/medium/high.'},
        {'role': 'user',   'content': 'My production database is down and customers are complaining.'},
    ],
    metadata={'lab': '9.2', 'pattern': 'openai-wrap'},  # Langfuse-only kwarg, ignored by upstream OpenAI SDK
)
print('Urgency:', completion.choices[0].message.content)
langfuse.flush()


**Tip.** The `metadata=` kwarg is intercepted by the wrapper and stripped before the request reaches OpenAI's API — safe to leave in production code. This is how you tag traces with `tenant_id`, `feature_flag`, `model_version`, etc.

### 5.4 Combine All Three Patterns in One Request

Real apps mix and match. The example below: an outer `@observe()`-decorated handler opens the trace, calls a LangChain chain (CallbackHandler picks it up as a child span), then drops to the wrapped OpenAI SDK for a follow-up classification step. All three calls land on **one** parent trace.

In [ ]:
from langfuse import observe, get_client
from langfuse.langchain import CallbackHandler
from langfuse.openai import openai
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

langfuse = get_client()

_chain = (
    ChatPromptTemplate.from_messages([
        ('system', 'You are a customer support agent. Reply in 1 sentence.'),
        ('human',  '{q}'),
    ])
    | ChatAnthropic(model='claude-haiku-4-5-20251001', temperature=0)
    | StrOutputParser()
)


@observe(name='handle-customer-question')
def handle(question: str) -> dict:
    handler = CallbackHandler()
    reply = _chain.invoke({'q': question}, config={'callbacks': [handler]})

    # Use the wrapped OpenAI SDK for a follow-up classification — different SDK, same trace.
    follow_up = openai.chat.completions.create(
        model='gpt-5-mini',
        messages=[
            {'role': 'system', 'content': 'Output one word: ROUTINE, URGENT, or ESCALATE.'},
            {'role': 'user',   'content': f'Customer asked: {question}\nAgent replied: {reply}'},
        ],
    )
    return {'reply': reply, 'priority': follow_up.choices[0].message.content.strip()}


result = handle('I have not received my order from 3 weeks ago.')
print(result)
langfuse.flush()
print('→ One trace, three child generations: LC chain → Anthropic call · OpenAI direct call.')


## 6. Custom Scoring — Push an LLM-as-Judge Score Onto a Trace

Tracing tells you *what happened*. Scoring tells you *whether it was any good*. The Langfuse **score** API attaches a numeric (or categorical) judgement to a trace, span, or generation — and those scores power the dashboard's quality views.

Pattern: (1) make a traced call, (2) capture the trace_id, (3) run a *separate* LLM-as-Judge pass on the output, (4) push the judge's score back onto the original trace. The judgement does not live inline with the live request — it can run async, in batch, or even days later.

In [ ]:
from langfuse import get_client
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langfuse.langchain import CallbackHandler

langfuse = get_client()

# Step 1 — make a traced call and remember the trace_id ────────────────────────────
with langfuse.start_as_current_span(name='customer-reply') as span:
    trace_id = span.trace_id
    handler = CallbackHandler()
    chain = (
        ChatPromptTemplate.from_messages([
            ('system', 'Reply to the customer in <=3 sentences. Be empathetic and clear.'),
            ('human',  '{question}'),
        ])
        | ChatAnthropic(model='claude-haiku-4-5-20251001', temperature=0)
        | StrOutputParser()
    )
    customer_question = 'I was double-charged on my last invoice — can you fix this today?'
    reply = chain.invoke({'question': customer_question}, config={'callbacks': [handler]})

print('Reply:', reply)
print('Trace id:', trace_id)

# Step 2 — separate LLM-as-Judge pass scores the reply ─────────────────────────────
judge_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Rate the customer-support reply on EMPATHY from 0.0 (cold/dismissive) to 1.0 '
               '(warm/genuine). Reply with just the number, nothing else.'),
    ('human',  'Customer asked: {q}\n\nAgent replied: {a}\n\nEmpathy score (0.0-1.0):'),
])
judge = judge_prompt | ChatAnthropic(model='claude-sonnet-4-5-20250929', temperature=0) | StrOutputParser()
raw_score = judge.invoke({'q': customer_question, 'a': reply}).strip()
try:
    empathy_score = float(raw_score)
except ValueError:
    empathy_score = 0.5  # judge gave non-numeric output; default to neutral
print(f'LLM-as-Judge empathy score: {empathy_score}')

# Step 3 — push the score onto the original trace ──────────────────────────────────
langfuse.create_score(
    name='empathy',
    value=empathy_score,
    trace_id=trace_id,
    comment=f'Judged by claude-sonnet-4-5 — raw: {raw_score!r}',
)
langfuse.flush()
print('Score posted. Filter the dashboard by score.empathy < 0.6 to find weak replies.')


### 6.1 Score Types — Numeric, Categorical, Boolean

Langfuse supports three score data types — pick deliberately so the dashboard aggregations make sense.

| Data type | Use it for | Example |
|---|---|---|
| `NUMERIC` | quality on a continuous scale; average over time | empathy=0.82 |
| `CATEGORICAL` | bucketed verdicts; bar-chart breakdowns | rating=GOOD/FAIR/POOR |
| `BOOLEAN` | pass/fail gates; alerts on flips | safety_pass=true |


In [ ]:
from langfuse import get_client

langfuse = get_client()

# Three score posts on the same trace, one per data type.
with langfuse.start_as_current_span(name='multi-score-demo') as span:
    demo_trace_id = span.trace_id

langfuse.create_score(name='empathy',     value=0.82,  trace_id=demo_trace_id, data_type='NUMERIC')
langfuse.create_score(name='rating',      value='GOOD', trace_id=demo_trace_id, data_type='CATEGORICAL')
langfuse.create_score(name='safety_pass', value=True,   trace_id=demo_trace_id, data_type='BOOLEAN')
langfuse.flush()
print('Three scores attached to trace', demo_trace_id)


## 7. Dashboard Views — Filter by `user_id`, by `latency > 5s`, by error

Once traces + scores are flowing, the Langfuse dashboard becomes the production diagnostic surface. Below are the three saved-view queries every on-call engineer ends up creating on day one.

### 7.1 Per-customer / per-tenant cost dashboard

Every trace can carry a `user_id` and a `session_id`. In the Langfuse UI: **Traces → filter → `userId equals CUST-00042`** — produces all traces for that customer in time order. Aggregate views in **Users** show total cost per user, average latency, and number of sessions.

Code-side: pass user_id to the trace via the SDK or via metadata (`metadata={'userId': ...}`).

### 7.2 Latency outlier hunt

**Traces → filter → `latency > 5000ms`** — surfaces every slow request. Pivot by `model` or by `metadata.tool` to find the offender. Pair with **Sessions → P95 latency** to spot a recent regression.

### 7.3 Error-only view

**Traces → filter → `level = ERROR`** — every trace whose root span errored out. The trace timeline lets you click into the failing span, read the exception, and copy the input that triggered it.

### 7.4 Score regressions over time

**Scores → filter `name = empathy` → group by week** → line chart. If empathy drops after a prompt deploy, you have a regression candidate that LangSmith / Langfuse Datasets (Lab 9.1 §8) can confirm with a controlled experiment.

In [ ]:
# Reproduce the 'per-user' tagging that powers the §7.1 dashboard view.
# Demonstrates: passing userId + sessionId via metadata so the Langfuse UI filters work.
from langfuse.langchain import CallbackHandler

for cust_id, question in [
    ('CUST-00042', 'Refund please.'),
    ('CUST-00091', 'Why is my charge pending?'),
    ('CUST-00042', 'Status of my refund?'),
]:
    handler = CallbackHandler()
    chain.invoke(
        {'question': question},
        config={
            'callbacks': [handler],
            'metadata': {
                'langfuse_user_id': cust_id,                # special-cased by the LC handler
                'langfuse_session_id': f'{cust_id}-2026-05-01',
                'tenant': 'retail-bank-IN',
            },
        },
    )

langfuse.flush()
print('Three traces dispatched. In Langfuse UI: Users → CUST-00042 → 2 sessions → costs sum-up.')


## 8. Prompt Management — Langfuse's Answer to LangSmith Hub

Langfuse has its own **Prompt Management** feature — push a versioned prompt by name, optionally tag it (`production`, `staging`), and pull it from your app at runtime. Same CI/CD pattern as LangSmith Hub from Lab 9.1 §9, just running entirely on your self-hosted infra.

Two flavours:
- **`text`** prompts — string templates with `{{variable}}` slots.
- **`chat`** prompts — message arrays (system / user / assistant) for chat models.


In [ ]:
from langfuse import get_client

langfuse = get_client()

# Create / update a chat prompt. Repeated calls with the same name produce new versions.
try:
    prompt = langfuse.create_prompt(
        name='eclerx-day9-customer-reply',
        type='chat',
        prompt=[
            {'role': 'system', 'content': 'You are a concise, empathetic customer-support agent.'},
            {'role': 'user',   'content': '{{question}}'},
        ],
        labels=['production'],   # the label your prod app pulls
        config={'temperature': 0.2, 'model_hint': 'claude-haiku-4-5'},
    )
    print(f'Prompt pushed: {prompt.name} v{prompt.version}')
except Exception as exc:
    print(f'Push failed: {exc}')


In [ ]:
# Pull the prompt back by name + label. In prod you'd cache this for 60 s to limit lookups.
try:
    pulled = langfuse.get_prompt('eclerx-day9-customer-reply', label='production')
    print(f'Pulled prompt v{pulled.version}, label=production')
    compiled = pulled.compile(question='How can I update my address?')
    print('Compiled prompt for invocation:')
    for msg in compiled:
        print(' ', msg)
except Exception as exc:
    print(f'Pull failed: {exc}')


**Cost-control tip.** `langfuse.get_prompt(...)` is cached client-side for 60 seconds by default — tunable via `cache_ttl_seconds`. Prod apps should set this generously (5-15 min) so the prompt-pull is essentially free, then call `langfuse.refresh_prompt(...)` from a webhook on label-change.

## 9. When to Use LangSmith vs Langfuse — Production Decision Rules

Treat the rules below as the team's house style. Edit them once for your client; never re-debate per project.

1. **Default for greenfield LangChain projects** → LangSmith Cloud. Zero-config, native, best DX.
2. **Multi-language stack (Python + JS + Go services)** → Langfuse. Better non-Python SDKs.
3. **Customer PII never leaves the bank's VPC** → Langfuse self-hosted. Hard requirement.
4. **The team is already using OTEL + ClickHouse** → Langfuse self-hosted. Drops into existing infra.
5. **Need long-term audit retention (7 years for SR 11-7)** → Langfuse self-hosted on cheap object storage.
6. **Procurement requires open-source license** → Langfuse (MIT).
7. **You want LangSmith Hub for prompt-version-as-CI-gate** → LangSmith Cloud (or self-hosted).
8. **You want the same tool for dev *and* prod** → either, but prefer the one your security team has already approved.

The hybrid pattern that actually ships: **LangSmith in dev/stage, Langfuse self-hosted in prod**, with a small shim that emits to both during the migration window.

---
## 10. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **Three integration patterns** | `@observe()` for plain Python · LangChain `CallbackHandler` for Runnables · `from langfuse.openai import openai` for raw SDK calls. |
| **Trace structure** | Trace → Spans → Generations + Scores. Every LLM call is a generation; every wrapper function is a span. |
| **Custom scoring** | `langfuse.create_score(trace_id=..., name='empathy', value=0.8)` — three data types: NUMERIC / CATEGORICAL / BOOLEAN. |
| **Prompt management** | `create_prompt` / `get_prompt(label='production')` — Langfuse's equivalent of LangSmith Hub, fully self-hostable. |
| **Self-host architecture** | Langfuse server (Node) + Postgres (metadata) + ClickHouse (events). Helm chart available; HA-ready. |
| **Decision matrix** | LangSmith for LangChain-first SaaS-OK teams; Langfuse self-hosted for regulated / OSS / data-residency requirements. |

**Next Lab:** Lab 9.3 — 3-Layer Agent Evaluation Hierarchy + Skills/Prompt Optimisation 🧪


## 11. Stretch Exercise (Optional)

1. Spin up Langfuse via `docker compose up` on your laptop, point your .env at `http://localhost:3000`, and confirm a trace appears in the local dashboard.
2. Add a *second* score on the trace from §6 — `factuality` — using a separate LLM-as-Judge prompt. Verify both scores show up under the trace.
3. Wrap the customer-support router agent from Lab 9.1 with the Langfuse `CallbackHandler` instead of LangSmith. Confirm every LangGraph node appears as a span.
4. Build a Langfuse Dataset (mirrors Lab 9.1 §8) and run an experiment via `langfuse.dataset(...).run(...)`. Compare the dashboards.
5. Write a small FastAPI endpoint that uses the `@observe()` decorator on the request handler and propagates `userId` from the JWT into the trace.


---

## Interview Preparation

The questions below mirror what client interviewers commonly ask about the topics in this lab. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. LangSmith vs Langfuse — what are the architectural differences?**

*Hint:* SaaS-only vs OSS+SaaS, single store vs split storage.

*Answer sketch:* **LangSmith** is a closed-source SaaS (with an enterprise self-host add-on) built around a single proprietary trace store, deeply native to LangChain. **Langfuse** is an MIT-licensed OSS server (Node) backed by **Postgres for metadata + ClickHouse for events**, with a Cloud option *and* a Helm chart for self-hosting. Langfuse splits hot-path event storage from metadata for cheap long retention; LangSmith hides storage from you. Both expose a similar Trace → Span → Generation hierarchy.

---

**Q2. When does self-hosting Langfuse beat using LangSmith Cloud?**

*Hint:* Hard regulatory / sovereignty requirements.

*Answer sketch:* When (a) customer PII / PHI cannot leave your VPC under GDPR / HIPAA / DPDP / RBI rules, (b) procurement mandates an OSS license, (c) you already run an OTEL + ClickHouse stack and want one fewer SaaS contract, (d) you need 7-year SR 11-7 audit retention on cheap object storage, or (e) you're behind an air-gap. For LangChain-first dev teams with no sovereignty constraints, LangSmith Cloud usually wins on DX.

---

**Q3. What does `@observe` do under the hood?**

*Hint:* Open span on enter, capture I/O, close span on exit, propagate context.

*Answer sketch:* It's a context-manager-flavoured decorator. On entry it opens a Langfuse *span* (or a new *trace* if none is active in the current async context), records the function name and arguments. On exit it captures the return value (or exception), computes latency, and closes the span. Nesting is automatic via Python's `contextvars`, so a decorated function calling another decorated function produces a parent → child tree without any explicit plumbing.

---

**Q4. How is a Langfuse trace structured — spans, generations, scores?**

*Hint:* Trace = root unit, span = work unit, generation = LLM call, score = judgement.

*Answer sketch:* A **Trace** is the top-level unit (one user request, one job, one experiment iteration). A **Span** is any work step inside the trace — a function call, a DB query, a retrieval. A **Generation** is a *specialised span* for an LLM call, with extra fields (model, prompt, completion, prompt_tokens, completion_tokens, cost). A **Score** is a numeric or categorical judgement attached to a trace / span / generation, used for quality dashboards and regression alerting. All four can carry metadata + tags for filtering.

---

**Q5. How do you push a quality score (e.g. from LLM-as-Judge) into a Langfuse trace?**

*Hint:* Capture the trace_id at request time, post the score later.

*Answer sketch:* 1) When the request runs, capture the trace_id (e.g. via `langfuse.start_as_current_span(name=...).trace_id` or from the LangChain handler). 2) Persist trace_id alongside the request output. 3) In a separate evaluation pass — sync or batch, even days later — run your LLM-as-Judge prompt over the request/response pair. 4) Call `langfuse.create_score(name='empathy', value=0.8, trace_id=trace_id)` to attach the judgement. The dashboard then aggregates scores by name/time/model.

---

**Q6. What's the data-residency story for regulated banks — why does it matter?**

*Hint:* Cross-border data transfer rules + sovereignty.

*Answer sketch:* GDPR Art 44, India's DPDP Act, RBI's data-localisation circular, UAE CBUAE Sandbox, Saudi SAMA — all restrict where customer data may be stored / processed. SaaS observability tools like LangSmith Cloud (US/EU regions) violate these for many tenants. Langfuse self-hosted lives entirely inside the bank's VPC (any region), so customer data never leaves the regulated perimeter. The audit trail itself becomes regulated data — keeping it on-prem closes the loop.

---

**Q7. How would you deploy Langfuse self-hosted in production (HA, scaling)?**

*Hint:* Helm chart + managed Postgres + ClickHouse cluster.

*Answer sketch:* Use the official **langfuse-k8s** Helm chart. Run **3+ API replicas** behind a load balancer, **2+ async workers** for ingestion, **Postgres in HA** (CNPG operator or RDS Multi-AZ) for metadata, and a **3-node ClickHouse cluster** (Altinity operator or ClickHouse Cloud) for events. Object storage (S3 / MinIO) for blob attachments. Add network policies so only the API/worker pods touch the databases, and rotate API keys via External Secrets. Standard mid-bank sizing handles ~50 M events/day.

---

**Q8. How do you correlate a Langfuse trace with a LangChain `RunnableConfig`?**

*Hint:* Pass the CallbackHandler in `config={'callbacks': [...]}` and use `metadata` for IDs.

*Answer sketch:* Instantiate `from langfuse.langchain import CallbackHandler; handler = CallbackHandler()`. Pass it via `config={'callbacks': [handler], 'metadata': {'langfuse_user_id': cust_id, 'langfuse_session_id': sess}}` on every Runnable invocation. The handler creates one trace per top-level runnable invocation, then nests spans for every child Runnable, LLM, retriever, and tool. The `metadata` keys with the `langfuse_` prefix are extracted and become first-class trace attributes — that's what powers the per-user dashboard view.

